In [1]:
# Cell 1: cài package (chạy 1 lần)
!pip install openai pypdf -q

In [2]:
# Cell 2: đọc file tài liệu (txt hoặc pdf)
from pathlib import Path
from pypdf import PdfReader

doc_path = Path("/home/jkl/Code/VLLM-PD/documents/test1.pdf")  # đổi path

if doc_path.suffix.lower() == ".pdf":
    reader = PdfReader(str(doc_path))
    text = "\n".join((p.extract_text() or "") for p in reader.pages)
else:
    text = doc_path.read_text(encoding="utf-8", errors="ignore")

# giới hạn độ dài để tránh vượt context
text = text[:12000]
print("Loaded chars:", len(text))

Loaded chars: 1745


In [3]:
# Cell 3: hỏi model local vLLM
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="EMPTY",
)

resp = client.chat.completions.create(
    model="llama3.1-8b-awq",
    temperature=0.2,
    messages=[
        {"role": "system", "content": "Bạn là trợ lý tóm tắt tài liệu."},
        {"role": "user", "content": f"Tóm tắt tài liệu sau thành 5 ý chính:\n\n{text}"}
    ],
)

print(resp.choices[0].message.content)

Dưới đây là tóm tắt tài liệu thành 5 ý chính:

**1. Mô hình tương tác bằng giọng nói đa ngôn ngữ cho robot humanoid (Unitree G1)**

Mô hình này cho phép tương tác bằng giọng nói tự nhiên, hỗ trợ cả tiếng Việt và tiếng Nhật, và được thiết kế để triển khai trên thiết bị Jetson Orin. Mục tiêu của mô hình này là giảm thiểu độ trễ và không phụ thuộc vào API dài hạn.

**2. Các hạn chế của hệ thống**

Hệ thống phải đáp ứng các hạn chế sau: 
- Sử dụng Unitree SDK cho Audio I/O
- Không có khả năng kiểm soát trực tiếp ALSA
- Giới hạn đa luồng CUDA
- Độ trễ không được vượt quá 2 giây
- Không phụ thuộc vào API dài hạn

**3. Cấu trúc hệ thống**

Cấu trúc hệ thống bao gồm các phần sau:
- Mic → Wakeword → VAD + ASR
- Intent Cache → LLM + RAG
- TTS → Audio Normalization → Speaker
- Client (Jetson) + Server (Cloud/Dev)

**4. Các thách thức về kỹ thuật**

Các thách thức về kỹ thuật bao gồm:
- Sự không phù hợp về tốc độ lấy mẫu
- Sự không khớp về thời gian buffer
- Sụt giảm và nhiễu tín hiệu
- Điều chỉnh